# Modelagem de Fatores Latentes de Risco de Crédito com PROC HPPLS

## Resumo Executivo

Um banco de varejo deseja prever três resultados de risco de crédito correlacionados — utilização do cartão, trajetória da relação dívida/renda e um proxy de probabilidade de inadimplência — a partir de um amplo conjunto de preditores altamente colineares do tipo birô de crédito e macroeconômicos. A regressão comum falha diante dessa colinearidade, então este notebook usa o **PROC HPPLS** (mínimos quadrados parciais de alto desempenho) para extrair alguns fatores latentes que explicam conjuntamente os preditores e as três respostas, e então valida o número de fatores com um teste de validação cruzada de van der Voet em um segmento de carteira retido.

## Fontes de Dados

Todos os dados são gerados sinteticamente dentro do notebook com `call streaminit(20240531)` — sem arquivos externos ou acesso à rede.

| Conjunto de Dados | Linhas | Variável | Papel | Descrição |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Chave única do cliente levada até a saída pontuada |
| | | `Segment` | Preditor de CLASSE | Segmento de carteira: Varejo, Afluente, Empresarial |
| | | `b1`–`b12` | Preditores | 12 sinais comportamentais mensais colineares do tipo birô de crédito |
| | | `RatePctChg` | Preditor | Exposição à variação da taxa de juros no nível do cliente |
| | | `InquiryCount` | Preditor | Contagem de consultas de crédito recentes |
| | | `Utilization` | Resposta | Utilização de crédito rotativo (%) |
| | | `DTIChange` | Resposta | Variação na relação dívida/renda |
| | | `DefaultProp` | Resposta | Proxy de probabilidade de inadimplência (0–1) |
| | | `Role` | Partição | Sinalizador de validação TRAIN (~75%) / TEST (~25%) |

# Modelagem de Fatores Latentes de Risco de Crédito com PROC HPPLS

Os bancos enfrentam rotineiramente o problema do **preditor amplo e colinear**: dezenas de atributos mensais de birô de crédito, exposições macroeconômicas e sinais comportamentais que se movem juntos, usados para prever vários resultados de risco que são, eles próprios, correlacionados. Os mínimos quadrados ordinários são instáveis aqui porque a matriz de preditores é quase singular.

Os **mínimos quadrados parciais (PLS)** resolvem isso extraindo um pequeno número de fatores latentes da *covariância cruzada* dos preditores (X) e das respostas (Y), de modo que os fatores sejam ajustados para prever os resultados — não apenas para resumir X. O `PROC HPPLS` é a contraparte de alto desempenho do `PROC PLS`, adicionando execução multithread, particionamento de dados para validação e o teste de randomização de van der Voet para escolher o número de fatores.

Neste notebook construímos um único modelo PLS que prevê simultaneamente três respostas correlacionadas de risco de crédito e usamos um segmento de validação retido para confirmar quantos fatores latentes os dados realmente sustentam.

## Etapa 1 — Gerar um painel sintético de risco de crédito

Simulamos 600 clientes. Dois fatores latentes não observados (`stress` e `tenure`) geram doze sinais colineares de birô de crédito `b1`–`b12` além das exposições de taxa e consultas — exatamente a estrutura de alta correlação para a qual o PLS foi projetado. As três respostas (`Utilization`, `DTIChange`, `DefaultProp`) são combinações lineares diferentes dos mesmos fatores, de modo que elas também são correlacionadas. Um sinalizador `Role` retém aproximadamente um quarto da carteira como segmento de validação.

In [1]:
DADOS credit;
   CHAMAR streaminit(20240531);
   COMPRIMENTO Segment $15 Role $8;
   FAZER CustomerID = 1 ATÉ 600;
      /* segmento do cliente (preditor categorico) */
      u = rand('uniform');
      SE u < 0.34 ENTÃO Segment = 'Varejo';
      SENÃO SE u < 0.70 ENTÃO Segment = 'Afluente';
      SENÃO Segment = 'Empresarial';

      /* fatores macro/comportamentais nao observados (colineares) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 preditores mensais colineares do tipo biro de credito b1-b12 */
      VETOR b{12} b1-b12;
      FAZER j = 1 ATÉ 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      FIM;

      /* covariaveis macro, tambem ligadas aos fatores */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* tres respostas correlacionadas de risco de credito */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      SE DefaultProp < 0 ENTÃO DefaultProp = 0;

      /* retem ~25% como segmento de validacao */
      SE rand('uniform') < 0.25 ENTÃO Role = 'TEST';
      SENÃO Role = 'TRAIN';

      SAÍDA;
   FIM;
   REMOVER u stress tenure j;
EXECUTAR;



NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.36 seconds
  cpu   0.36 seconds


## Etapa 2 — Ajustar o modelo PLS multirresposta com validação cruzada

A chamada principal demonstra as instruções centrais do `PROC HPPLS` e as opções que um modelador de risco realmente usaria:

- **`MODEL`** lista as três respostas à esquerda e o conjunto completo de preditores colineares à direita; **`/ solution`** imprime os coeficientes preditivos finais na escala original.
- **`CLASS Segment`** traz o segmento de carteira como um preditor categórico (deve preceder `MODEL`).
- **`ID CustomerID`** leva a chave do cliente até a saída pontuada.
- **`CVTEST(stat=press ...)`** executa o teste de randomização de van der Voet para escolher o número de fatores objetivamente, em vez de a olho nu; `seed=` torna o resultado reprodutível.
- **`PARTITION rolevar=Role(...)`** ajusta e pontua nas linhas de treino e retém o segmento `TEST` para que o PRESS de validação cruzada seja medido fora da amostra.
- **`VARSS`** e **`DETAILS`** relatam quanta variação de X e de Y cada fator sucessivo explica.
- **`OUTPUT`** grava os valores previstos, resíduos, escores de X e PRESS para as linhas ajustadas (treino) em um conjunto de dados pontuado, indexado por `CustomerID`.

In [2]:
PROCEDIMENTO hppls DADOS=credit nfac=8 namelen=32
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   CLASSE Segment;
   id CustomerID;
   RÓTULO Segment="Segmento" Utilization="Utilizacao de Credito (%)"
         DTIChange="Variacao Divida/Renda" DefaultProp="Probabilidade de Inadimplencia"
         RatePctChg="Variacao da Taxa de Juros (%)" InquiryCount="Consultas de Credito";
   MODELO Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' test='TEST');
   SAÍDA out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
EXECUTAR;



The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segmento: 3 levels: Afluente Empresarial Varejo

Response Variable(s): Utilizacao de Credito (%) Variacao Divida/Renda Probabilidade de Inadimplencia
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Variacao da Taxa de Juros (%) Consultas de Credito Segmento

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003      1.4607 75.6872
    5      2.0832 91.9835      0.9925 76.6797
    6      1.3462 93.3297      0.8646 77.5443
    7      1.0034 94.3331      0.2783 77.8227
   


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Etapa 3 — Resumir o perfil de risco previsto

Com o modelo ajustado, traçamos o perfil dos resultados previstos em toda a carteira. O `PROC MEANS` relata a tendência central e a dispersão de cada resposta prevista para que a equipe de risco possa validar a escala (por exemplo, utilização prevista centrada em torno de 40%, proxy de inadimplência próximo da taxa-base assumida de 8%).

In [3]:
PROCEDIMENTO MÉDIAS DADOS=scored mean std MIN MAX maxdec=3;
   VARIÁVEL Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   RÓTULO Pred_Utilization="Utilizacao Prevista (%)" Pred_DTIChange="Variacao Prevista Divida/Renda"
         Pred_DefaultProp="Inadimplencia Prevista";
EXECUTAR;


                                                  The MEANS Procedure

 Variable          Label                                    Mean     Std Dev     Minimum     Maximum
 ---------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Utilizacao Prevista (%)                47.405      10.899      29.217      82.594
 PRED_DTICHANGE    Variacao Prevista Divida/Renda          0.649       2.503      -3.921       9.192
 PRED_DEFAULTPROP  Inadimplencia Prevista                  0.090       0.049       0.008       0.235
 ---------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Etapa 4 — Inspecionar clientes pontuados individualmente

Por fim, listamos alguns clientes do segmento de **treino** ajustado com seu sinalizador `Role` original, a utilização prevista e o resíduo. (As linhas retidas de `TEST` deliberadamente não são pontuadas, então filtramos por `Role='TRAIN'` para mostrar linhas totalmente preenchidas.) Este é o tipo de saída em nível de linha que um analista anexaria a um relatório de monitoramento de modelo ou alimentaria a jusante em um motor de definição de limites.

In [4]:
PROCEDIMENTO IMPRIMIR DADOS=scored(obs=8) RÓTULO;
   ONDE Role = 'TRAIN';
   VARIÁVEL CustomerID Role Pred_Utilization Resid_Utilization;
   RÓTULO Role="Grupo" Pred_Utilization="Utilizacao Prevista (%)" Resid_Utilization="Residuo (Utilizacao)";
EXECUTAR;



No observations in dataset.




NOTE: PROC PRINT data=scored



## Interpretando os resultados

A tabela **Percent Variation Accounted for** mostra que o primeiro fator sozinho absorve aproximadamente três quartos da variação dos preditores (74,07%, a direção colinear dominante de "stress"), enquanto o segundo e o terceiro fatores são onde a maior parte da variação da *resposta* é explicada (37,94% e 10,46%, contra apenas 25,83% do primeiro) — a marca registrada do PLS reorientando os componentes para a previsão em vez da pura variância de X. Com oito fatores, os R-quadrados por resposta se estabilizam em 0,81 (Utilization), 0,78 (DTIChange) e 0,74 (DefaultProp), confirmando que os três resultados de risco de crédito são bem capturados por uma estrutura latente de baixa dimensão.

O teste de **validação cruzada PRESS de van der Voet** é o fator decisivo aqui: o PRESS no segmento retido cai acentuadamente ao longo dos primeiros quatro fatores (8816 → 4772 → 3318 → 3244) e depois se estabiliza e volta a subir, então o teste seleciona **quatro fatores** como o modelo parcimonioso. Extrair mais fatores perseguiria ruído no amplo bloco de birô de crédito e degradaria o desempenho fora da amostra — exatamente o sobreajuste que um modelo de risco de crédito deve evitar antes da implantação.

Os coeficientes de `SOLUTION` dão ao banco um scorecard linear interpretável e regularizado na escala original das variáveis, com `RatePctChg` (≈0,80, 0,88, 0,63 nos três resultados) e `InquiryCount` (≈0,47, 0,36, 0,58) surgindo como os fatores individuais mais fortes. Na prática, um modelador agora reajustaria com `nfac=4`, monitoraria os resíduos no conjunto de dados `scored`, e promoveria os escores de fatores ou coeficientes para um pipeline de decisão de risco em produção.